# Data Poisoning against a Fraud Detector


## Scenario

A FinTech company retrains its fraud model every night on transactions labeled by
analysts and customer chargebacks. An attacker cannot touch the model directly, but can
influence *which labels end up in the training set* (e.g. fraudulent transactions that are
never reported, or crafted transactions that look like normal customers).




> **Note:** Create one virtual env and install these dependencies there:
>
> - `python -m venv .ART`
> - `pip install -r requirements-ART.txt`

## 1-Import Libraries

In [ ]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score


In [ ]:
warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED) 

> The `SEED` doesn’t produce a single fixed number, it fixes the whole sequence of “random” choices. So each time we run the experiment, we get the same result.

## 2-Data


We generate 10 numeric features (amount, hour, merchant risk score, velocity...).
Real fraud is far rarer (approximately 0.1-0.2%). But here, for the sake of our class, we use **10% fraud** so a small lab runs fast and the effect is visible. Features are scaled to [0, 1] because ART's SVM attack needs clip values.

In [ ]:
X, y = make_classification(
    n_samples=1500, n_features=10, n_informative=6, n_redundant=2,
    weights=[0.90, 0.10], class_sep=2.0, flip_y=0.0, random_state=SEED,
)

feature_names = ["amount", "hour", "merchant_risk", "velocity", "distance_from_home",
                 "card_age", "n_tx_24h", "avg_amount_30d", "foreign_tx", "device_risk"]

scaler = MinMaxScaler().fit(X)

X = scaler.transform(X)

In [ ]:

# Three-way split:
# 1️⃣ train (what the attacker can pollute).
# 2️⃣ validation (trusted, used by the defender).
# 3️⃣ test (held-out audit set; never touched by attacker or defense).

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.6, stratify=y, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.6, stratify=y_tmp, random_state=SEED)

print("DONE. We have created our sample data 😎")
print(f"train={len(y_tr)} (fraud={y_tr.sum()}), val={len(y_val)} (fraud={y_val.sum()}), test={len(y_te)} (fraud={y_te.sum()})")
print("😎😎😎😎😎")

In [ ]:
# We'll dump this dataset on our local system to see what's there exactly.
import pandas as pd
import os

os.makedirs("data", exist_ok=True)

df = pd.DataFrame(X_tr, columns=feature_names)

df["is_fraud"] = y_tr

df.to_csv("data/train_preview.csv", index=False)

## 3-Baseline model and evaluation


**Why not accuracy?**

Predicting "legit" for everything gives ~90% accuracy here and catches zero fraud.
We track **recall** (fraud caught), **precision** (alert quality), F1 and PR-AUC.

 The cell below, trains a fraud detector on the clean data and measures how good it is.

Note that “How good” means how well the model catches fraud without raising false alarms.

The cell measures that with two main numbers:
   - Recall (0.74): the model catches about 74 out of every 100 real frauds, so 26 slip through.
   - Precision (0.98): when it flags a transaction as fraud, it is right 98% of the time.

Higher is better for both. The attacks later in the lab try to push recall down so fraud gets done.

### A short definition of three metrics:

- **Precision**: of all the transactions the model flagged as fraud, how many really were fraud. High precision means few false alarms.

- **F1**: one score that combines precision and recall. It is high only when both are high, so a model can’t look good by maxing one and ignoring the other.

- **PR-AUC**: the model gives every transaction a fraud score. PR-AUC measures how well it ranks real frauds above legit ones, across all possible thresholds. It doesn’t depend on one cutoff: 1.0 is perfect ranking, and a random model scores about the fraud rate (0.10 here).

In [ ]:
def make_model():
    return SVC(kernel="linear", C=1.0)

def evaluate(model, X_eval, y_eval, name=""):
    pred = model.predict(X_eval)
    score = model.decision_function(X_eval)

    out = dict(
        precision=precision_score(y_eval, pred, zero_division=0),
        recall=recall_score(y_eval, pred, zero_division=0),
        f1=f1_score(y_eval, pred, zero_division=0),
        pr_auc=average_precision_score(y_eval, score),
    )
    if name:
        print(f"{name} ;;; " + " 📝📝📝 ".join(f"{k}={v:.3f}" for k, v in out.items()))
    return out

baseline = make_model().fit(X_tr, y_tr)

base_metrics = evaluate(baseline, X_te, y_te, "Baseline (clean data)")

## 🚩Now let's go for two attacks and put our black hoodies on 🧑‍💻

## 4-Attack A: label flipping


**The cheapest attack**:
Relabel a fraction of **fraud** rows as legitimate
(for example, fraud that was never reported, so it is stored as "*normal*").

In [ ]:
# 🚩 The below funct, flip "rate" fraction of the FRAUD labels (1 turns to 0)

# [3,6,8,23]  ===> y = 0 

def flip_labels(y, rate, rng):
    y_p = y.copy()  # copy y so the original labels stay unchanged
    fraud_idx = np.where(y == 1)[0] # positions (indexes) of all fraud rows, where y == 1
    n_flip = int(rate * len(fraud_idx)) # how many fraud labels to flip (rate x number of frauds)
    chosen = rng.choice(fraud_idx, size=n_flip, replace=False)  # randomly pick n_flip fraud positions, no repeats
    y_p[chosen] = 0   # change the chosen labels from 1 (fraud) to 0 (legit)
    return y_p  # return the poisoned labels

rates = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]

flip_results = []

for r in rates:
    y_poison = flip_labels(y_tr, r, np.random.default_rng(SEED))

    m = make_model().fit(X_tr, y_poison)
    
    flip_results.append(evaluate(m, X_te, y_te))

    print(f"flip {int(r*100):>3}% of fraud labels 😎===>😎 recall={flip_results[-1]['recall']:.3f}  precision={flip_results[-1]['precision']:.3f}")

In [ ]:
rate = 0.3
y_before = y_tr
y_after = flip_labels(y_tr, rate, np.random.default_rng(SEED))

# print before and after
print("fraud labels before:", y_before.sum(), "| after:", y_after.sum())
changed = np.where(y_before != y_after)[0]

print("flipped rows:", len(changed))

print("their row indexes:", changed)

# print(df[df["flipped"]].head())

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot([r * 100 for r in rates], [m["recall"] for m in flip_results], "o-", label="recall")
plt.plot([r * 100 for r in rates], [m["precision"] for m in flip_results], "s--", label="precision")
plt.xlabel("% of fraud labels flipped to 'legit'")
plt.ylabel("metric on clean test set")
plt.title("Label flipping attack")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 5-Attack B: ART `PoisoningAttackSVM`


**The idea in one sentence:** instead of just relabeling existing rows (Attack A), the attacker *crafts* poison points and places them exactly where they hurt the model the most. Here, Attacker builds fake transactions and makes them as harmful as possible.

### How it works
1. The attacker starts with a fraud-like transaction and gives it the label "legit".
2. The attack then moves that point a little at a time (**gradient ascent**; just means moving in the direction that increases something, here the error).
3. After each move it checks: *if the SVM is retrained with this point, does the error on the validation set go up?*
4. It keeps moving the point in the direction that makes the error grow, until it reaches the most harmful position it can find.

> The result is a point that looks like a plausible transaction but is placed to push the model’s decision boundary the wrong way. Add 12 or 24 of them to the training data, and the model learns to miss fraud.

In [ ]:
# ART's SVM attack needs a small, balanced-enough subset and one-hot labels.
# Here sub somes from subset

sub = np.concatenate([
    rng.choice(np.where(y_tr == 0)[0], 80, replace=False),
    np.where(y_tr == 1)[0][:40],
])
Xs, ys = X_tr[sub], y_tr[sub]
Xv, yv = X_val[:150], y_val[:150]

def one_hot(v):
    return np.eye(2)[v]

clean_svc = make_model().fit(Xs, ys)
clean_metrics_sub = evaluate(clean_svc, X_te, y_te, "Subset baseline (clean)")

The below cell, makes 24 poisoned transactions, adds them to the training data, and checks how much damage they do.

1. **Set up the attack.** It wraps the SVM for ART and creates the attack object, giving it the training data and the trusted validation data.
2. **Pick starting points.** It takes 24 real fraud rows (`seeds`) at random. Each will become a poison point labeled "legit".
3. **Craft the poison.** In the loop, `attack.poison(...)` moves each point to the position that raises the validation error the most.
4. **Poison the training set.** It adds the 24 crafted points, all labeled legit, to the clean 120-row subset, then trains a new model on the result and tests it on the clean test set (`art_metrics`).
5. **Control.** It does the same with the 24 original fraud rows, unmoved but relabeled legit (`naive_metrics`), so you can see whether ART's optimization did better than the simple approach.

The two printed lines are the results to compare.

> An attacker could run code like this on their own computer to search for harmful points.

In [ ]:
from art.estimators.classification.scikitlearn import ScikitlearnSVC # ART wrapper that lets ART work with a scikit-learn SVM
from art.attacks.poisoning import PoisoningAttackSVM  # ART's optimized poisoning attack for SVMs

N_POISON = 24  # how many poison points the attacker creates
np.random.seed(32)
t0 = time.time()

# Train a normal SVC on the clean subset, then wrap it for ART.
# probability=True is needed because ART asks the model for probabilities.
# clip_values=clip keeps every feature between 0 and 1 (no impossible transactions).
# clf stands for classifier 

clip = (0.0, 1.0)
art_clf = ScikitlearnSVC(model=SVC(kernel="linear", C=1.0, probability=True,
                                   random_state=SEED).fit(Xs, ys), clip_values=clip)

# Build the attack. It needs the wrapped model, the training data (white-box; remind from previous video)
# and a validation set to measure how much damage a point causes.
# step=0.1: size of each move; eps=0.001: stopping tolerance; max_iter=100: maximum moves per point.

attack = PoisoningAttackSVM(
    art_clf, step=0.1, eps=0.001,
    x_train=Xs, y_train=one_hot(ys), x_val=Xv, y_val=one_hot(yv), max_iter=100, verbose=False,
)

# Choose the starting points: copies of real FRAUD rows that will become poison.
fraud_rows = Xs[ys == 1]  # all fraud rows in the subset
seeds = fraud_rows[rng.choice(len(fraud_rows), N_POISON, replace=False)]  # randomly pick 24 of them

# Craft the poison: move each starting point to where it hurts the model most.
poison_pts = []  # list that collects the crafted points. pts comes from points

for i, p in enumerate(seeds):  # go through the 24 starting points one by one
    pt, _ = attack.poison(np.array([p]), y=one_hot(np.array([0])))  # move the point; the label we want is "legit" (0)
    poison_pts.append(pt[0])  # keep the moved point

# 🚩 "attack.poison" takes our starting point and returns the crafted, harmful version.

poison_X = np.array(poison_pts)  # turn the list into one array of 24 rows
poison_y = np.zeros(N_POISON, dtype=int)  # all 24 poison points carry the label "legit"

print(f"Crafted {N_POISON} poison points") # write sth on the console
print(f"=====================")
print(f"=====================")

# Poison the training data: clean subset + 24 poison points, then train and test.

X_pois = np.vstack([Xs, poison_X])  # stack features: clean rows on top, poison below
y_pois = np.concatenate([ys, poison_y])  # join the labels the same way
art_poisoned = make_model().fit(X_pois, y_pois)  # train a new model on the poisoned data
art_metrics = evaluate(art_poisoned, X_te, y_te, "After ART SVM poisoning")  # test on clean test data

# Control: same number of poison points, but WITHOUT optimization (original fraud rows, labeled legit).
naive_X = np.vstack([Xs, seeds])   # clean rows + the 24 unmoved fraud rows
naive_y = np.concatenate([ys, np.zeros(N_POISON, dtype=int)])  # label those 24 rows "legit"
naive_model = make_model().fit(naive_X, naive_y)   # train on this simpler poisoned set
naive_metrics = evaluate(naive_model, X_te, y_te, "Just use the naive fraud")  # test it and compare with ART

### What happened in the above cell?

- First, we create one SVC model.
- Second, we wrapped it in ART.
- Third, use this wrapped model to craft poisonous data.
- Fourth, add these bad data to the main dataset.
- Fifth, retrain the model from scratch with these new dataset.

## 6-Defenses; data sanitization before retraining


### ART `RONIDefense`
Here we measure how much **adding** each training point changes accuracy on a trusted validation set. We'll
reject points with a clearly negative impact.


> RONI comes from `Reject On Negative Impact`

In [ ]:
from sklearn.metrics import accuracy_score
from art.defences.detector.poison import RONIDefense

# RONI compares one-hot labels with probability output, so reduce both to class ids
def acc_from_probs(y_true_onehot, y_pred_probs):
    return accuracy_score(y_true_onehot.argmax(axis=1), y_pred_probs.argmax(axis=1))

is_poison_truth = np.concatenate([np.zeros(len(ys)), np.ones(N_POISON)]).astype(bool)

def score_defense(flagged, name):
    caught = (flagged & is_poison_truth).sum()
    false_pos = (flagged & ~is_poison_truth).sum()
    print(f"{name}: flagged {flagged.sum()}/{len(flagged)} | poison caught {caught}/{N_POISON} | clean points wrongly removed {false_pos}")

roni_clf = ScikitlearnSVC(model=SVC(kernel="linear", C=1.0, probability=True, random_state=32), clip_values=clip)
np.random.seed(32)
roni = RONIDefense(
    roni_clf, X_pois, one_hot(y_pois), Xv, one_hot(yv),
    perf_func=acc_from_probs, pp_cal=0.2, pp_quiz=0.2, calibrated=True,
)
_, roni_clean = roni.detect_poison()
roni_flagged = ~np.array(roni_clean).astype(bool)
score_defense(roni_flagged, "ART RONI")

keep = ~roni_flagged # explanation at the end
roni_metrics = evaluate(make_model().fit(X_pois[keep], y_pois[keep]), X_te, y_te, "After RONI sanitization")


# roni_flagged = np.array([True, False, True, False])
# keep = ~roni_flagged
# keep = [False, True, False, True]

### Some notes regarding RONI

RONI is expensive, and it did not work in this small setup, but it is not useless in general.

 - Expensive: it retrains the model many times to test each point’s impact.
 - Weak here: with about 140 rows and an accuracy metric, no single point stands out.
 - Not useless in general: it was designed for settings where individual points matter more.

### Label-consistency (k-NN) sanitization
Flag a training point if its label disagrees with the majority of its
k nearest neighbours (leave-one-out). Our poison points are fraud-like rows labelled "legit", so they sit
inside a cluster of fraud neighbours and stand out.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def knn_flag(X, y, k=7):
    flagged = np.zeros(len(y), dtype=bool) # one True/False per row, all False at first
    for i in range(len(y)): 
        others = np.arange(len(y)) != i # at the end of this cell
        knn = KNeighborsClassifier(n_neighbors=k).fit(X[others], y[others])  # build a neighbor voter from all the other points
        flagged[i] = knn.predict(X[i:i + 1])[0] != y[i] # flag point i if the neighbors' vote differs from its label
    return flagged   

knn_flagged = knn_flag(X_pois, y_pois, k=7)
score_defense(knn_flagged, "kNN sanitizer (k=7)")

knn_metrics = evaluate(make_model().fit(X_pois[~knn_flagged], y_pois[~knn_flagged]), X_te, y_te, "After kNN sanitization")


# np.arange(5)        
# # [0, 1, 2, 3, 4] (the row numbers)
# np.arange(5) != 2   
# # [True, True, False, True, True]

# knn.predict(X[i:i + 1])[0] will predict the label of ith row
# then compares or check against y[i]. If they are equall, returns True and otherwise returns False.

# X_pois[~knn_flagged], y_pois[~knn_flagged] means keep only the rows that were not flagged.

# Finnaly, in flagged, those True items are “suspected poison” and should be considered.


**Why this works here, and why it will not always:**

The attacker used *flipped labels on fraud-like points*,
which a neighbourhood vote exposes immediately. A **clean-label** or **backdoor** attacker (correct labels, subtle
trigger) would evade this check. Defenses must match the threat model.

### What we have done?

For each training point, the code asks its 7 nearest neighbors “what label should this point have?” If they disagree with its actual label, the point is flagged as suspicious.

Poison points look like fraud but are labeled “legit”, so their fraud-like neighbors disagree and they get flagged. The model is then retrained without the flagged points and tested.

## 7-Audit summary (what goes into the security report)

In [ ]:
rows = [
    ("Clean (subset)", clean_metrics_sub),
    ("ART SVM poisoned", art_metrics),
    ("Naive flip (control)", naive_metrics),
    ("ART RONI defense", roni_metrics),
    ("kNN sanitizer", knn_metrics),
]
# 👇 <24 means the text is left-aligned in a field 24 characters wide
print(f"{'Scenario':<24}{'Precision':>10}{'Recall':>10}{'F1':>10}{'PR-AUC':>10}")
for n, m in rows:
    print(f"{n:<24}{m['precision']:>10.3f}{m['recall']:>10.3f}{m['f1']:>10.3f}{m['pr_auc']:>10.3f}")

plt.figure(figsize=(8, 3.8))
labels = [r[0] for r in rows]
plt.bar(labels, [r[1]["recall"] for r in rows], color=["steelblue", "indianred", "darkorange", "orchid", "forestgreen"]) # type: ignore
plt.ylabel("Fraud recall (clean test set)")
plt.title("Recall under poisoning and defenses")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()